<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## An LP Checked with KKT

**Minimize the largest clock error and verify its active multipliers.**

Retain the recorded errors. Use one constant correction, remove calibration and the correction bound, and replace squared error by largest absolute error.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Turn an absolute-error limit into linear inequalities

Choose a constant correction $q$ (min) for errors $d=(-2,-1,-1,0)$ min. Remaining errors are $d_i+q$, $i=1,\ldots,4$. Introduce an auxiliary error limit $z$ (min):

$$
\begin{aligned}
\min_{q,z}\quad &z\\
\text{subject to}\quad &d_i+q-z\le0,\quad -(d_i+q)-z\le0\quad (i=1,\ldots,4),\\
&-z\le0.
\end{aligned}
$$

The objective and constraints are affine, so this is a **linear program (LP)**. The physical decision is $q$; $z$ expresses the comparison rule. A smaller $z$ matters only if every error lies in $[-z,z]$.

The extreme errors $q-2$ and $q$ remain 2 min apart, so fitting them requires $2z\ge2$. At $(q,z)=(1,1)$ the errors $(-1,0,0,1)$ fit. This attains the lower bound and proves optimality.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-1_constrained_optimization_kkt/assets/lp_compact.svg" alt="The smallest feasible error limit as a function of correction is a V with minimum at correction 1 and limit 1." width="512" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Verify KKT at the two active limits

At $(1,1)$ the active residuals are $2-q-z$ and $q-z$. All other residuals are negative, so their multipliers are zero. Call the active multipliers $\alpha,\beta\ge0$:

$$
L=z+\alpha(2-q-z)+\beta(q-z).
$$

Stationarity gives $-\alpha+\beta=0$ and $1-\alpha-\beta=0$, hence $\alpha=\beta=0.5$. Feasibility, signs, complementarity, and stationarity all hold. This LP is convex, so KKT also certifies the global minimum.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · The objective determines the preferred correction

For the original data, constant-correction least squares also selects $q=1$, but its score is $2$ min², not the largest error $1$ min. Agreement is not general. If only the final recorded error changes to $3$, the data become $(-2,-1,-1,3)$:

| Rule | Selected correction (min) | Squared error (min²) | Largest error (min) |
|:---|---:|---:|---:|
| Least squares | $0.25$ | $14.75$ | $3.25$ |
| Largest absolute error | $-0.5$ | $17$ | $2.5$ |

Each choice wins under its own rule. LP and QP describe forms; both can be analyzed with KKT.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Interactive comparison

The $q$ slider changes the physical correction, while the $z$ slider changes only the proposed error limit. All four points must fit in the shaded band. The candidate $(q,z)=(1,1)$ reaches the smallest feasible limit; lowering $z$ further makes it infeasible.

In [ ]:
import sys
import numpy as np
import matplotlib

if sys.platform != "emscripten":
    try:
        matplotlib.use("widget", force=True)
    except (RuntimeError, ValueError):
        from matplotlib.backends import backend_registry
        backend_registry._clear()
        matplotlib.use("widget", force=True)
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

BLUE, TEAL, ORANGE, PURPLE, GRAY = "#2878b5", "#168578", "#e78b24", "#8854a5", "#707070"
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11})

observed_errors = np.array([-2., -1., -1., 0.])  # min

def evaluate_limit(correction, limit):
    errors = observed_errors + correction
    residuals = np.r_[errors-limit, -errors-limit, -limit]
    return {"errors": errors, "objective": limit,
            "feasible": bool(np.all(residuals <= 1e-9)),
            "largest_error": float(np.max(np.abs(errors)))}

In [ ]:
figure, axis = plt.subplots(figsize=(5.6, 4.8))
figure.subplots_adjust(left=0.15, right=0.96, top=0.90, bottom=0.40)
positions = np.arange(1, 5)
points = axis.scatter(positions, observed_errors, color=BLUE, zorder=3)
band = axis.axhspan(-1, 1, color=TEAL, alpha=0.15)
axis.set(xlabel="Reading i", ylabel="Remaining error (min)", xticks=positions,
         ylim=(-3, 3), title="Every error must fit inside the proposed limit")
axis.grid(alpha=0.25)
status = figure.text(0.15, 0.24, "", va="top")
q_slider = Slider(figure.add_axes([0.25, 0.12, 0.58, 0.035]), "q (min)",
                  0, 2, valinit=0, valstep=0.05, color=BLUE)
z_slider = Slider(figure.add_axes([0.25, 0.05, 0.58, 0.035]), "z (min)",
                  0.5, 2.5, valinit=1, valstep=0.05, color=TEAL)

def update(_):
    global band
    result = evaluate_limit(q_slider.val, z_slider.val)
    points.set_offsets(np.c_[positions, result["errors"]])
    points.set_color([BLUE if abs(error)<=z_slider.val+1e-9 else GRAY
                      for error in result["errors"]])
    band.remove()
    band = axis.axhspan(-z_slider.val, z_slider.val, color=TEAL, alpha=0.15)
    status.set_text(f"{'Feasible' if result['feasible'] else 'Rejected'}; proposed z = {z_slider.val:.2f} min\n"
                    f"Largest physical error = {result['largest_error']:.2f} min")
    figure.canvas.draw_idle()

figure._sliders = [q_slider, z_slider]
for slider in figure._sliders:
    slider.on_changed(update)
update(None)
plt.show()